# Topic: ML: Cross-Validation

## Definition (30-second explanation)
* Cross-Validation (CV) evaluates model performance by splitting a dataset into $k$ equal parts (folds).
* The model is trained $k$ times; each time, $k-1$ folds are used for training, and the remaining 1 fold is held out for testing.
* The final performance is the average score across all $k$ iterations, ensuring every single data point is used for testing exactly once.

## Why Interviewers Ask This
* **Robustness Check:** Assesses your understanding of variance in model evaluation (a single random split can be "lucky" or "unlucky").
* **Data Leakage Awareness:** Tests if you know how to properly apply preprocessing steps (like scaling) *inside* the CV loop using Pipelines.
* **Class Imbalance:** Checks if you default to `StratifiedKFold` for classification to prevent folds with zero minority class samples.

## Core Concepts
* **K-Fold CV:** The standard approach for regression, splitting data into $k$ consecutive or shuffled blocks.
* **Stratified K-Fold CV:** For classification; guarantees that the target class distribution (e.g., 90% positive, 10% negative) is preserved in every single fold.
* **Bias-Variance Trade-off in $k$:** A lower $k$ (e.g., 3) has higher bias and lower variance. A higher $k$ (e.g., 10 or Leave-One-Out) has lower bias but higher variance and computation time.
* **Stability Metric:** The standard deviation (`std`) of the CV scores tells you how stable the model is across different data subsets (lower is better).

## When to Use
* When working with small to medium datasets where you cannot afford to "waste" data on a single hold-out validation set.
* When comparing the performance of multiple algorithms or tuning hyperparameters.
* Whenever an interviewer explicitly asks for a "reliable" or "robust" model evaluation strategy.

## Advantages
* Provides a much more reliable, lower-variance estimate of real-world model performance compared to a simple train-test split.
* Maximizes data utility, as 100% of the dataset is eventually used for both training and validation.

## Limitations
* Computationally expensive: running a 10-fold CV takes roughly 10 times longer than a single train-test split.
* Not suitable for massive datasets (millions of rows) where a simple hold-out set is statistically sufficient and CV would take days.
* Standard CV breaks temporal logic; time-series data requires `TimeSeriesSplit`.

## Common Comparisons
* **Cross-Validation vs. Train-Test Split:** Train-test is fast but high-variance (dependent on the random seed). CV is slow but low-variance and highly reliable.
* **K-Fold vs. Stratified K-Fold:** K-Fold is for continuous targets (regression). Stratified is for categorical targets (classification) to maintain class balance.

## Common Interview Traps
* **The Preprocessing Leakage Trap:** Fitting a `StandardScaler` or `SimpleImputer` on the *entire* dataset before running `cross_val_score`. This leaks information from the validation folds into the training folds.
* **Ignoring Group Structures:** Using standard CV on medical data where multiple rows belong to the same patient. The same patient might end up in both train and test folds. (Fix: Use `GroupKFold`).

## Python / SQL Syntax
```python
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Define strategy and model
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression()

# Execute cross-validation
scores = cross_val_score(model, X, y, cv=cv_strategy, scoring='accuracy')

print(f"Mean CV Score: {scores.mean():.4f}")
print(f"Std CV Score: {scores.std():.4f}")
```

## Important Formula
* **CV Score Average:** $CV_{score} = \frac{1}{k} \sum_{i=1}^{k} Score_i$

## 45-Second Interview Answer
"Cross-validation is a technique to robustly evaluate model performance by splitting data into $k$ folds, training on $k-1$, and testing on the remaining fold, repeating this until every fold has been a test set. It provides a more reliable, lower-variance metric than a single train-test split. For classification tasks, I always use `StratifiedKFold` to maintain class balance. Most importantly, I wrap my preprocessing steps and model in a scikit-learn `Pipeline` before passing it to the CV function to absolutely guarantee no data leakage occurs across folds."

## Example Questions:

### Q1:
**Question:** What is the difference between train-test split and k-fold cross-validation? When would you prefer one over the other?

**Ideal Interview Answer:**
Train-test split divides the data once, making it computationally fast but highly dependent on the random seed, leading to a high-variance performance estimate. K-fold CV splits the data $k$ times, testing on every single data point exactly once, which provides a much more robust and reliable estimate at the cost of compute time. I prefer train-test split for massive datasets or quick prototyping, and k-fold CV for small datasets, strict model comparison, or hyperparameter tuning.

**Common Mistakes:**
* Stating that CV is "always better" without acknowledging the significant computational overhead.

**Likely Interviewer Follow-up:**
"If you have a dataset of 10 million rows, why might k-fold CV actually be a bad idea?"

### Q2:
**Question:** What is the trade-off between k=5 and k=10 in k-fold cross-validation?

**Ideal Interview Answer:**
The choice between k=5 and k=10 is fundamentally a bias-variance and computational trade-off. With k=10, you train on 90% of the data each time, which lowers the pessimistic bias of your performance estimate, but the folds are highly correlated, which can slightly increase the variance of the estimate. Furthermore, k=10 requires training the model 10 times, taking twice as long as k=5. In practice, k=5 or k=10 are industry standards that balance this well.

**Common Mistakes:**
* Stating that a higher $k$ always reduces variance (a common misconception; as folds overlap more, variance of the estimate can actually increase).

**Likely Interviewer Follow-up:**
"What happens if we set $k$ equal to the total number of rows in the dataset?"

### Q3:
**Question:** What is Leave-One-Out (LOO) cross-validation and when is it appropriate?

**Ideal Interview Answer:**
Leave-One-Out is the extreme edge case of K-Fold where $k$ equals the total number of samples ($N$) in the dataset. The model trains on $N-1$ samples and tests on exactly 1 sample, repeating this $N$ times. It is only appropriate for extremely small datasets (e.g., fewer than 50-100 rows) where you cannot afford to leave out any data for training. For normal or large datasets, it is far too computationally expensive.

**Common Mistakes:**
* Suggesting LOO for datasets with thousands of rows, which would require training thousands of separate models.

**Likely Interviewer Follow-up:**
"Does Leave-One-Out CV involve stratification? Why or why not?"

### Q4:
**Question:** How do you use cross-validation with preprocessing steps without data leakage?

**Ideal Interview Answer:**
To prevent data leakage, you must ensure that preprocessing (like scaling, imputing, or PCA) is fit *only* on the training folds, not the validation fold. If you scale the entire dataset before CV, the training folds learn the global mean/variance, leaking validation data. I handle this programmatically by chaining my preprocessors and model together using a scikit-learn `Pipeline`, and then passing the entire `Pipeline` object to the `cross_val_score` function. 

**Common Mistakes:**
* Saying "I will scale X_train inside a loop" (while technically true if manually looping, interviewers want to hear the keyword `Pipeline`).

**Likely Interviewer Follow-up:**
"Can you write out the basic syntax for how you would construct that Pipeline?"

### Q5:
**Question:** What is the difference between cross_val_score and cross_validate in sklearn?

**Ideal Interview Answer:**
`cross_val_score` is a simplified function that returns a single array of evaluation scores (like accuracy) for each fold. `cross_validate` is a more robust function that returns a dictionary. It allows you to evaluate multiple metrics at once (e.g., accuracy, precision, and recall simultaneously) and also returns the fit times and score times for each fold, which is highly useful for production profiling.

**Common Mistakes:**
* Not knowing `cross_validate` exists, which forces you to run `cross_val_score` three separate times to get three different metrics, tripling the compute time.

**Likely Interviewer Follow-up:**
"If you wanted to retrieve the actual trained models from each fold to inspect their feature importances, which function would you use?"